In [1]:
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Scaling
from sklearn.preprocessing import StandardScaler

# KNN
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score

In [2]:
df_train = pd.read_csv("train_KNN.csv")
df_test = pd.read_csv("test_KNN.csv")

df_train.info()
df_test.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived

In [3]:
df_train.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [4]:
mp = 100 * (df_train.isnull().sum() / len(df_train))
mp = mp[mp > 0].sort_values()
mp

,0
Embarked,0.224467
Age,19.865320
Cabin,77.104377


In [5]:
# Fill missing Age with median
df_train["Age"] = df_train["Age"].fillna(df_train["Age"].median())

# Drop rows where Embarked is missing
df_train = df_train.dropna(subset=["Embarked"])

# Drop Cabin (too many missing values)
df_train = df_train.drop(["Cabin"], axis=1)

df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    int64  
 1   Survived     889 non-null    int64  
 2   Pclass       889 non-null    int64  
 3   Name         889 non-null    object 
 4   Sex          889 non-null    object 
 5   Age          889 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    object 
 9   Fare         889 non-null    float64
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 83.3+ KB


In [6]:
# Check missing values percentage
mpt = 100 * (df_test.isnull().sum() / len(df_test))
mpt = mpt[mpt > 0].sort_values()
mpt

,0
Fare,0.239234
Age,20.574163
Cabin,78.229665


In [7]:
# Drop rows where Fare is missing
df_test = df_test.dropna(subset=["Fare"])

# Fill missing Age with median
df_test["Age"] = df_test["Age"].fillna(df_test["Age"].median())

# Drop Cabin
df_test = df_test.drop(["Cabin"], axis=1)

df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 417 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  417 non-null    int64  
 1   Survived     417 non-null    int64  
 2   Pclass       417 non-null    int64  
 3   Name         417 non-null    object 
 4   Sex          417 non-null    object 
 5   Age          417 non-null    float64
 6   SibSp        417 non-null    int64  
 7   Parch        417 non-null    int64  
 8   Ticket       417 non-null    object 
 9   Fare         417 non-null    float64
 10  Embarked     417 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 39.1+ KB


In [8]:
df_train.drop(["PassengerId","Name","Ticket"], axis=1, inplace=True)
df_test.drop(["PassengerId","Name","Ticket"], axis=1, inplace=True)

df_train.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    object 
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Fare      889 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 62.5+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 417 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  417 non-null    int64  
 1   Pclass    417 non-null    int64  
 2   Sex       417 non-null    object 
 3   Age       417 non-null    float64
 4   SibSp     417 non-null    int64  
 5   Parch     417 non-null    int64  
 6   Fare      417 non-null    float6

In [10]:
df_train["Survived"] = df_train["Survived"].apply(str)
df_train["Pclass"] = df_train["Pclass"].apply(str)

df_test["Survived"] = df_test["Survived"].apply(str)
df_test["Pclass"] = df_test["Pclass"].apply(str)

In [11]:
df_train_num = df_train.select_dtypes(exclude="object")
df_train_obj = df_train.select_dtypes(include="object")

df_test_num = df_test.select_dtypes(exclude="object")
df_test_obj = df_test.select_dtypes(include="object")

In [12]:
df_train_obj = pd.get_dummies(df_train_obj, drop_first=True)
df_test_obj = pd.get_dummies(df_test_obj, drop_first=True)

In [13]:
Final_train_df = pd.concat([df_train_num, df_train_obj, df_train["Survived"]], axis=1)
Final_test_df = pd.concat([df_test_num, df_test_obj, df_test["Survived"]], axis=1)

Final_train_df.head()
Final_test_df.head()

,Age,SibSp,Parch,Fare,Survived_1,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,34.5,0,0,7.8292,False,False,True,True,True,False,0
1,47.0,1,0,7.0000,True,False,True,False,False,True,1
2,62.0,0,0,9.6875,False,True,False,True,True,False,0
3,27.0,0,0,8.6625,False,False,True,True,False,True,0
4,22.0,1,1,12.2875,True,False,True,False,False,True,1


In [14]:
X_train = Final_train_df.drop("Survived", axis=1)
y_train = Final_train_df["Survived"]

X_test = Final_test_df.drop("Survived", axis=1)
y_tes = Final_test_df["Survived"]

In [15]:
scaler = StandardScaler()
scaler.fit(X_train)

scaled_X_train = scaler.transform(X_train)
scaled_X_test = scaler.transform(X_test)

In [16]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(scaled_X_train, y_train)

y_pred = knn.predict(scaled_X_test)

accuracy_score(y_tes, y_pred)

0.9928057553956835

### KNN Interpretation

The KNN model achieved an accuracy of approximately 99.28%, which is very high.
This could indicate overfitting or data leakage between training and testing datasets.
Although the model performs very well on the test data, it may not generalize well to unseen data.

In [17]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(scaled_X_train, y_train)

y_pred_lr = lr.predict(scaled_X_test)

lr_accuracy = accuracy_score(y_tes, y_pred_lr)

lr_accuracy

1.0

In [18]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

dt_accuracy = accuracy_score(y_tes, y_pred_dt)

dt_accuracy

1.0

In [19]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["KNN", "Logistic Regression", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_tes, y_pred),
        lr_accuracy,
        dt_accuracy
    ]
})

results

,Model,Accuracy
0,KNN,0.992806
1,Logistic Regression,1.000000
2,Decision Tree,1.000000


## Final Findings

The KNN model recorded an accuracy of around 99.28%, whereas the Logistic Regression and Decision Tree models recorded 100% accuracy. The extremely high accuracy in these results almost invariably points towards either data leakage within the dataset or presence of very strong relationships between features and target. Models have likely memorized the training data, so their ability to perform on new unseen data might be limited.

On one hand, Logistic Regression is a straightforward and interpretable model, Decision Tree is versatile but prone to overfitting. KNN is quite good, though its performance heavily relies on distance metrics and feature scaling. Closing Statement:

Despite the extremely high accuracies of all models, Logistic Regression remains the most trustworthy one because it is simple and easy to interpret; however, on the whole the results must be considered as indicative of possible overfitting and should be used very cautiously.